# HadISD Data Download Notebook

This notebook will help you download a subset of the HadISD dataset directly from the Met Office website. The data will be stored in a user-specified directory (or a sensible default), and extracted for further processing (e.g., conversion to Zarr).

- **Source:** [HadISD v3.4.0.2023f WMO_000000-029999.tar.gz](https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/data/WMO_000000-029999.tar.gz)
- **Instructions:**
    1. Set the download directory (or use the default).
    2. Download the data using Python's `requests` (recommended for stability) or `wget`.
    3. Extract the `.tar.gz` archive.
    4. The extracted files will be ready for use in the next notebook (conversion to Zarr).

> **Note:** Download size is large. Ensure you have sufficient disk space and a stable internet connection.

### Set Path to Download Directory
If you want to change the download directory, you can do so here. Otherwise, it will default to a folder named "HadISD_data" in your home directory.

In [ ]:
import os
from pathlib import Path

# Set the download directory (user can change this if desired)
default_dir = Path.home() / "HadISD_data"
download_dir = os.environ.get("HADISD_DOWNLOAD_DIR", str(default_dir))
download_dir = Path(download_dir)
download_dir.mkdir(parents=True, exist_ok=True)

print(f"Data will be downloaded to: {download_dir}")

### Download HadISD Data
The following code will download the HadISD data files. Some files take longer to download than others, depending on time of day. You can change the `download_number` to download different WMO datasets to suit your needs.

Alternatively you can change the WMO number to download different datasets. The full list of available data can be found here:
https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/download.html

In [ ]:
import requests
from tqdm.auto import tqdm

# --- Download HadISD data by WMO number range ---
# Provide a sample list of WMO number ranges. Users can find more at the official HadISD download page.
sample_wmo_ranges = [
    "000000-029999",
    "080000-099999",
    "200000-249999",
    "720000-721999",
]

# User sets the WMO number range to download
wmo_range = "200000-249999"  # Change this to the desired WMO range (see markdown cell for more options)

wmo_str = f"WMO_{wmo_range}"
url = f"https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/data/{wmo_str}.tar.gz"
tar_name = f"{wmo_str}.tar"
filename = download_dir / tar_name

# Download with resume support and correct progress bar
headers = {}
initial_pos = 0
if filename.exists():
    initial_pos = filename.stat().st_size
    headers['Range'] = f'bytes={initial_pos}-'
    mode = 'ab'
else:
    mode = 'wb'

response = requests.get(url, stream=True, headers=headers)
total = int(response.headers.get('content-length', 0)) + initial_pos

with open(filename, mode) as f, tqdm(
    desc=f"Downloading {filename.name}",
    total=total,
    initial=initial_pos,
    unit='B', unit_scale=True, unit_divisor=1024
) as bar:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)
            bar.update(len(chunk))

print(f"Download complete: {filename}")

Download complete: /Users/joelmiller/HadISD_data/WMO_200000-249999.tar


In [ ]:
import tarfile

# Extract the tar.gz file
extract_dir = download_dir / tar_name.replace('.tar', '') 
  # Change this to the desired extraction directory         
extract_dir.mkdir(exist_ok=True)

with tarfile.open(filename, "r:gz") as tar:
    tar.extractall(path=extract_dir)

print(f"Extraction complete. Files are in: {extract_dir}")

/var/folders/v1/7wypnx5x11qgv022zwx5p3g00000gn/T/ipykernel_85648/3880857257.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


Extraction complete. Files are in: /Users/joelmiller/HadISD_data/WMO_200000-249999


In [ ]:
import gzip
import shutil

# --- Create subfolder for netcdf ---
netcdf_dir = download_dir / tar_name.replace('.tar', '')  / "netcdf"
netcdf_dir.mkdir(parents=True, exist_ok=True)

# Move extracted .nc files into netcdf_dir after extraction
for gz_path in extract_dir.glob('*.nc.gz'):
    nc_path = gz_path.with_suffix('')  # Remove .gz extension
    with gzip.open(gz_path, 'rb') as f_in, open(nc_path, 'wb') as f_out:
        f_out.write(f_in.read())
    print(f"Extracted: {nc_path}")
    gz_path.unlink()  # Delete the .gz file after extraction
    print(f"Deleted: {gz_path}")
    # Move the .nc file to netcdf_dir
    shutil.move(str(nc_path), netcdf_dir / nc_path.name)
    print(f"Moved: {nc_path} -> {netcdf_dir / nc_path.name}")

print("All .nc.gz files have been extracted, cleaned up, and moved to the netcdf directory.")